# GeoDevelop Processing Lab
Run the backend on your Colab GPU. This is a prototype; inspect results and review model/data terms. Your inputs are transmitted to this runtime. Keep tokens in Colab Secrets.

1. Select a GPU runtime with Python 3.12 or newer.
2. Add HF_TOKEN in Secrets and enable notebook access (approved SAM 3 access required).
3. Run setup, restart the session after installation, then continue.
4. Authorize Earth Engine yourself for satellite acquisition.
5. The API is private to this runtime until you explicitly open a tunnel. No cloud purchase is needed for these notebook steps.


In [ ]:
import sys, subprocess
import torch
print(sys.version, torch.__version__)
assert sys.version_info >= (3,12)
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print(torch.cuda.get_device_name(0))
subprocess.run(['git','clone','--depth','1','https://github.com/TALHANASEER-GIS/Geo-develop-AI.git','/content/geodevelop'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-r','/content/geodevelop/backend/requirements.txt','transformers','accelerate','huggingface_hub','einops','setuptools<81'],check=True)
subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/sam3.git','/content/sam3'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','/content/sam3'],check=True)
print('Restart session now so installed binary packages and editable paths are loaded.')


In [ ]:
import os,secrets,sys
from google.colab import userdata
from huggingface_hub import hf_hub_download
try:
    token=userdata.get('HF_TOKEN')
    checkpoint=hf_hub_download('facebook/sam3','sam3.pt',token=token)
except Exception:
    raise RuntimeError('Check Colab HF_TOKEN access and approved SAM3 weights access.') from None
finally:
    token=None
os.environ['SAM3_CHECKPOINT']=checkpoint
os.environ['SAM3_REPO']='/content/sam3'
os.environ['GEODEVELOP_DATA']='/content/geodevelop-data'
os.environ['GEODEVELOP_API_KEY']=secrets.token_urlsafe(32)
os.environ['GEODEVELOP_LLM_PROVIDER']='transformers'
os.environ['GEODEVELOP_TRANSFORMERS_MODEL']='Qwen/Qwen2.5-1.5B-Instruct'
sys.path.insert(0,'/content/geodevelop')
print('Private runtime access key (paste only into your dashboard connection form):')
print(os.environ['GEODEVELOP_API_KEY'])


In [ ]:
# Optional satellite authorization: complete Google's sign-in yourself.
import ee
ee.Authenticate()
os.environ['EE_PROJECT']='geo-develop-ai'
ee.Initialize(project=os.environ['EE_PROJECT'])
print('Earth Engine authorized.')


In [ ]:
# Start the API inside this runtime. Do not expose it without the bearer key.
import threading,uvicorn
from backend.app import app
server=uvicorn.Server(uvicorn.Config(app,host='127.0.0.1',port=8765,log_level='warning'))
thread=threading.Thread(target=server.run,daemon=True);thread.start()
print('Backend starting on runtime loopback port 8765. A tunnel is still required for GitHub Pages access.')


## Connect the dashboard
To connect GitHub Pages, open an authenticated HTTPS tunnel to port 8765 using a trusted provider. This exposes the API to the Internet; keep the bearer key private, restrict GEODEVELOP_ORIGINS, and stop the tunnel when finished. This notebook deliberately does not create public exposure automatically. In Processing Lab, enter the tunnel URL and private runtime key.

Large inputs are processed in tiles. Colab disk/runtime state is temporary. Download reports and GeoTIFFs before the session ends. Road and flood outputs are candidates; recorded human evidence is required for confirmed counts or damage labels.
